In [ ]:
%pip install datasets soundfile pydub numpy huggingface_hub

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os
import csv
import numpy as np
from datasets import load_dataset
from pydub import AudioSegment

# --- Konfigurace ---
OUTPUT_DIR = "/content/drive/MyDrive/datasets"
TARGET_HOURS = 10
SPLIT = "train"

LANGUAGES = {
    "cs": "czech",
    "de": "german",
    "en": "english",
    "es": "spanish",
    "et": "estonian",
    "fi": "finnish",
    "fr": "french",
    "hr": "croatian",
    "hu": "hungarian",
    "it": "italian",
    "lt": "lithuanian",
    "nl": "dutch",
    "pl": "polish",
    "ro": "romanian",
    "sk": "slovak",
    "sl": "slovene",
}
# -------------------

In [ ]:
def numpy_to_mp3(audio_array: np.ndarray, sample_rate: int, output_path: str):
    """Převede numpy audio pole na MP3 soubor."""
    if audio_array.dtype in (np.float32, np.float64):
        audio_int16 = (audio_array * 32767).astype(np.int16)
    else:
        audio_int16 = audio_array.astype(np.int16)

    audio_segment = AudioSegment(
        data=audio_int16.tobytes(),
        sample_width=2,
        frame_rate=sample_rate,
        channels=1,
    )
    audio_segment.export(output_path, format="mp3", bitrate="128k")


def get_existing_progress(lang_dir: str):
    """Zjistí kolik hodin a souborů už je staženo z existujícího transcripts.csv."""
    csv_path = os.path.join(lang_dir, "transcripts.csv")
    if not os.path.exists(csv_path):
        return 0, 0.0  # count, total_seconds

    count = 0
    total_seconds = 0.0
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            count += 1
            total_seconds += float(row["duration_s"])
    return count, total_seconds

In [ ]:
import time

def download_language(lang_code: str, lang_name: str, target_hours: float):
    """Stáhne VoxPopuli audio pro jeden jazyk."""
    lang_dir = os.path.join(OUTPUT_DIR, lang_name)
    os.makedirs(lang_dir, exist_ok=True)
    target_seconds = target_hours * 3600

    # Kontrola existujícího progressu
    existing_count, existing_seconds = get_existing_progress(lang_dir)
    if existing_seconds >= target_seconds:
        print(f"  {lang_name}: už staženo {existing_seconds/3600:.1f}h ({existing_count} souborů) — přeskakuji")
        return

    print(f"\n{'='*60}")
    print(f"  Stahování: {lang_name} ({lang_code})")
    if existing_count > 0:
        print(f"  Pokračování od: {existing_count} souborů, {existing_seconds/3600:.1f}h")
    print(f"  Cíl: {target_hours}h → {lang_dir}/")
    print(f"{'='*60}")

    # Streaming — nestahuje celý dataset
    dataset = load_dataset(
        "facebook/voxpopuli",
        lang_code,
        split=SPLIT,
        streaming=True,
        trust_remote_code=True,
    )

    csv_path = os.path.join(lang_dir, "transcripts.csv")
    write_header = existing_count == 0

    total_seconds = existing_seconds
    file_index = existing_count
    skip_count = existing_count
    start_time = time.time()

    with open(csv_path, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        if write_header:
            writer.writerow(["filename", "transcript", "speaker_id", "gender", "duration_s"])

        for i, sample in enumerate(dataset):
            # Přeskočení už stažených vzorků
            if i < skip_count:
                continue

            audio = sample["audio"]
            audio_array = np.array(audio["array"], dtype=np.float32)
            sample_rate = audio["sampling_rate"]
            duration_s = round(len(audio_array) / sample_rate, 2)

            # Uložení MP3
            filename = f"{file_index:05d}.mp3"
            output_path = os.path.join(lang_dir, filename)
            numpy_to_mp3(audio_array, sample_rate, output_path)

            # Zápis do CSV
            transcript = sample.get("normalized_text") or sample.get("raw_text", "")
            speaker_id = sample.get("speaker_id", "")
            gender = sample.get("gender", "")
            writer.writerow([filename, transcript, speaker_id, gender, duration_s])
            csvfile.flush()

            total_seconds += duration_s
            file_index += 1

            # Progress
            if file_index % 10 == 0:
                elapsed = time.time() - start_time
                new_files = file_index - existing_count
                new_audio_s = total_seconds - existing_seconds
                files_per_s = new_files / elapsed
                audio_h_per_min = (new_audio_s / 3600) / (elapsed / 60)
                remaining_s = (target_seconds - total_seconds) / (new_audio_s / elapsed) if new_audio_s > 0 else 0
                remaining_min = remaining_s / 60
                print(
                    f"  [{file_index} souborů] {total_seconds/3600:.1f}h / {target_hours}h "
                    f"| {files_per_s:.1f} souborů/s "
                    f"| {audio_h_per_min:.2f}h audia/min "
                    f"| zbývá ~{remaining_min:.0f} min"
                )

            # Dosažen cíl
            if total_seconds >= target_seconds:
                break

    elapsed = time.time() - start_time
    print(f"  {lang_name}: {file_index} souborů, {total_seconds/3600:.1f}h (za {elapsed/60:.1f} min)")

In [ ]:
print(f"VoxPopuli Download — {len(LANGUAGES)} jazyků, {TARGET_HOURS}h na jazyk")
print(f"Výstup: {OUTPUT_DIR}/\n")

for lang_code, lang_name in LANGUAGES.items():
    try:
        download_language(lang_code, lang_name, TARGET_HOURS)
    except Exception as e:
        print(f"  Chyba u {lang_name}: {e}")

print(f"\nHotovo!")

## Souhrn

In [ ]:
print(f"{'Jazyk':<15} {'Souborů':>8} {'Hodin':>8}")
print("-" * 33)

for lang_code, lang_name in LANGUAGES.items():
    lang_dir = os.path.join(OUTPUT_DIR, lang_name)
    count, total_seconds = get_existing_progress(lang_dir)
    if count > 0:
        print(f"{lang_name:<15} {count:>8} {total_seconds/3600:>7.1f}h")